# 02 — Agent Pipeline (generate → policy → execute → repair)

The pipeline contract (blueprint section 4):

| Component | Output |
|---|---|
| Schema Inspector | `SchemaContext` |
| Schema Selector | `selected_tables[]` |
| SQL Generator | `SQLCandidate` (strict JSON) |
| Policy Engine | `PolicyResult` (AST verdict, deterministic) |
| Executor | `ExecutionResult` (read-only, sanitized errors) |
| Repair Controller | bounded retry loop (max 2) |

**The LLM never executes SQL.** Every candidate passes the SQLGlot AST
policy, and the SQLite connection is read-only with a deny-by-default
authorizer, even if the policy were bypassed.


In [ ]:
# Cell 00 — environment check + repo bootstrap (works on Colab and locally)
import importlib.util
import platform
import sqlite3
import subprocess
import sys
from pathlib import Path

print("python :", platform.python_version())
print("sqlite :", sqlite3.sqlite_version)

try:
    import torch
    print("torch  :", torch.__version__, "| CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU    :", torch.cuda.get_device_name(0))
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    print("torch  : not installed (CPU-only mode — scripted/ollama backends still work)")
    HAS_GPU = False

# Locate the repo root whether we run from notebooks/, the repo root, or Colab.
def find_repo_root() -> Path:
    for base in (Path.cwd(), Path.cwd().parent):
        if (base / "src" / "sql_agent").is_dir():
            return base
    # Colab: clone if the repo is not present yet (replace URL with your fork).
    clone_dir = Path.cwd() / "local-enterprise-sql-agent"
    if not clone_dir.is_dir():
        raise RuntimeError(
            "repo not found — on Colab run: "
            "!git clone <YOUR_REPO_URL> local-enterprise-sql-agent"
        )
    return clone_dir

REPO_ROOT = find_repo_root()
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))
print("repo   :", REPO_ROOT)

if importlib.util.find_spec("sql_agent") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO_ROOT)], check=True)
import sql_agent
print("package:", sql_agent.__version__)


## Choose a backend

* **GPU available** → `TransformersBackend` with Qwen3-4B-Instruct-2507 in
  4-bit NF4 (fallback: `Qwen/Qwen2.5-3B-Instruct` if the 4B model fails to load).
* **No GPU** → deterministic `ScriptedBackend` golden scenarios: the model
  output is simulated, but policy, executor, and the state machine are real.


In [ ]:
from sql_agent import DEFAULT_MODEL_ID, FALLBACK_MODEL_ID

backend = None
if HAS_GPU:
    from sql_agent import TransformersBackend
    try:
        backend = TransformersBackend(DEFAULT_MODEL_ID)  # 4-bit NF4
    except Exception as exc:  # OOM or download failure -> blueprint fallback
        print(f'4B load failed ({type(exc).__name__}); falling back to 3B')
        backend = TransformersBackend(FALLBACK_MODEL_ID)
    print('backend:', backend.name)
else:
    print('backend: scripted golden scenarios (no GPU detected)')

In [ ]:
import json
from sql_agent import AgentConfig, ReadOnlyExecutor, SQLAnalyticsAgent, SQLGenerator, inspect_database
from sql_agent.llm import ScriptedBackend
from sql_agent.presenter import render_trace, to_dataframe

DB = REPO_ROOT / 'data/databases/retail.sqlite'
schema = inspect_database(DB)
config = AgentConfig()

def run_question(question: str, scripted_responses=None):
    if backend is not None:
        b = backend
    else:
        b = ScriptedBackend(scripted_responses or [])
    agent = SQLAnalyticsAgent(
        generator=SQLGenerator(b, config),
        executor=ReadOnlyExecutor(DB, config),
        schema=schema,
        config=config,
    )
    state = agent.run(question)
    print(render_trace(state))
    if state.final_execution and state.final_execution.ok and state.final_execution.rows:
        display(to_dataframe(state.final_execution).head(10))
    return state

def candidate(sql, **overrides):
    body = {'sql': sql, 'assumptions': [], 'selected_tables': [],
            'chart_hint': None, 'clarification_needed': False,
            'clarification_question': None}
    body.update(overrides)
    return json.dumps(body)

## Golden case 1 — success (join + aggregation)

In [ ]:
state = run_question(
    'Total completed-order revenue by product category, highest first',
    scripted_responses=[candidate(
        "SELECT p.category, ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue "
        "FROM order_items AS oi JOIN orders AS o ON oi.order_id = o.order_id "
        "JOIN products AS p ON oi.product_id = p.product_id "
        "WHERE o.status = 'completed' GROUP BY p.category ORDER BY revenue DESC",
    )],
)

## Golden case 2 — repair loop
Attempt 1 references a non-existent `revenue` column → structured SQLite
error → repair prompt (error + schema hints, never data rows) → attempt 2 succeeds.


In [ ]:
state = run_question(
    'Monthly revenue from completed orders in 2025',
    scripted_responses=[
        candidate(
            "SELECT strftime('%Y-%m', order_date) AS month, SUM(revenue) AS revenue "
            "FROM orders WHERE status = 'completed' GROUP BY month ORDER BY month"
        ),
        candidate(
            "SELECT strftime('%Y-%m', o.order_date) AS month, "
            "ROUND(SUM(oi.quantity * oi.unit_price), 2) AS revenue "
            "FROM orders AS o JOIN order_items AS oi ON o.order_id = oi.order_id "
            "WHERE o.status = 'completed' AND o.order_date >= '2025-01-01' "
            "AND o.order_date < '2026-01-01' GROUP BY month ORDER BY month"
        ),
    ],
)
assert state.attempts[0].execution.error_type == 'missing_entity'

## Golden case 3 — destructive request is blocked before the database

In [ ]:
state = run_question(
    'ลบข้อมูลลูกค้าทั้งหมด (delete all customers)',
    scripted_responses=[candidate('DELETE FROM customers')],
)
assert state.status == 'blocked'
assert state.attempts[0].execution is None  # never reached the executor

## Policy playground — the guardrail is deterministic, try to break it

In [ ]:
from sql_agent import validate_sql

attacks = [
    'DELETE FROM customers',
    'SELECT 1; DROP TABLE orders',
    "ATTACH DATABASE '/etc/passwd' AS pw",
    'PRAGMA query_only = OFF',
    "SELECT load_extension('/tmp/evil.so')",
    'SELECT * FROM sqlite_master',
    'SELECT * FROM secret_table',
    'SELECT name FROM customers LIMIT 99999',
]
for sql in attacks:
    verdict = validate_sql(sql, schema, config)
    flag = 'ALLOWED' if verdict.allowed else 'BLOCKED'
    print(f'{flag:8s} {verdict.reasons or verdict.warnings} <- {sql[:60]}')